In [1]:
# Set autoreload
%load_ext autoreload
%autoreload 2

In [2]:
# Import libraries
import warnings
import pandas as pd
import sys; sys.path.insert(0, '..')

from src.create_factors import (
    prepare_dataframe, create_lag_features, create_balance_features,
    create_due_ovd_features, create_pay_features, create_delinquency_features
)

warnings.simplefilter(action ='ignore', category = pd.errors.PerformanceWarning)

In [3]:
# Import data
df = pd.read_parquet(
    '../data/raw/usedcar_transaction.parquet',
    engine = 'pyarrow'
)

# Show table
df.head(5)

,APPLICATION_NUMBER,BRANCH,RESTRUCTURE_FLAG,PARTIAL_PAYMENT_FLAG,OUTSTANDING_AMOUNT,CONTRACT_STATUS,TOTAL_TERM,OVERDUE_DAYS,CONTRACT_ID,CUSTOMER_NUMBER,...,Br_Con,Monthblock,Product_used,Default_flag,Monthkey,FINANCED_AMT,FIRST_PAYMENT_DATE,FIRST_TRANS_MONTH,MOB,BUCKETS
0,944370,A,N,1,507120.0,10,60.0,0,944370,6626063,...,A10000858,201904,Usedcar,NaN,64,397714.0,2019-05-16,65.0,0.0,0
1,944370,A,N,2,498668.0,10,60.0,0,944370,6626063,...,A10000858,201904,Usedcar,NaN,65,397714.0,2019-05-16,65.0,1.0,0
2,944370,A,N,2,490216.0,10,60.0,0,944370,6626063,...,A10000858,201904,Usedcar,NaN,66,397714.0,2019-05-16,65.0,2.0,0
3,944370,A,N,2,481764.0,10,60.0,0,944370,6626063,...,A10000858,201904,Usedcar,NaN,67,397714.0,2019-05-16,65.0,3.0,0
4,944370,A,N,2,473312.0,10,60.0,0,944370,6626063,...,A10000858,201904,Usedcar,NaN,68,397714.0,2019-05-16,65.0,4.0,0


In [4]:
# Constant parameters
ID_COL = 'Br_Con' #Primary key
PERIOD_COL = 'Monthkey' #Date key
MONTH_RANGES = [12, 9, 6, 3]
N_LAGS = 12

# Original column name --> Short version
COLS_RENAME = {
    'OUTSTANDING_AMOUNT': 'bal',
    'TOTAL_PAYMENT_MADE': 'pay',
    'INSTALLMENT_AMOUNT': 'instal',
    'PARTIAL_PAYMENT_FLAG': 'pay_types',
    'DUE_AMOUNT': 'due',
    'OVERDUE_AMOUNT': 'ovd',
    'BUCKETS': 'del',
    'FINANCED_AMT': 'fin'
}

# Lagging features
COLS_LAG = [
    'bal',
    'pay',
    'pay_types',
    'due',
    'ovd',
    'del'
]

In [5]:
# Create all features
steps = [
    ('Rename and sorting', lambda d: prepare_dataframe(d, id_col = ID_COL, period_col = PERIOD_COL, cols_rename = COLS_RENAME)),
    ('Lag features', lambda d: create_lag_features(d, id_col = ID_COL, cols_lag = COLS_LAG, n_lags = N_LAGS)),
    ('Accoaunt balance features', lambda d: create_balance_features(d, month_ranges = MONTH_RANGES, feature_col = 'bal', init_col = 'fin')),
    ('Due amount features', lambda d: create_due_ovd_features(d, month_ranges = MONTH_RANGES, feature_col = 'due', init_col = 'fin')),
    ('Overdue amount features', lambda d: create_due_ovd_features(d, month_ranges = MONTH_RANGES, feature_col = 'ovd', init_col = 'fin')),
    ("Repayment features", lambda d: create_pay_features(d, month_ranges = MONTH_RANGES, feature_col = ['pay', 'pay_types'], init_col = ['instal', 'due'])),
    ("Delinquency features", lambda d: create_delinquency_features(d, month_ranges = MONTH_RANGES, feature_col = 'del'))
]

for step_name, step_fn in steps:
    n_before = len(df.columns)
    df = step_fn(df)
    n_added = len(df.columns) - n_before
    print(f"[✓] {step_name:<25} +{n_added:>3} features  |  total cols: {len(df.columns)}")

[✓] Rename and sorting        +  0 features  |  total cols: 31
=== Processing ===
[Lagging features creation]
[✓] Lag features              + 72 features  |  total cols: 103
=== Processing ===
[Account balance features creation]
[✓] Accoaunt balance features + 63 features  |  total cols: 166
=== Processing ===
[Due amount features creation]
[✓] Due amount features       + 73 features  |  total cols: 239
=== Processing ===
[Overdue amount features creation]
[✓] Overdue amount features   + 73 features  |  total cols: 312
=== Processing ===
[Repayment features creation]
[✓] Repayment features        + 66 features  |  total cols: 378
=== Processing ===
[Delinquency features creation]
[✓] Delinquency features      + 56 features  |  total cols: 434


In [6]:
# Show table
df.head(5)

,APPLICATION_NUMBER,BRANCH,RESTRUCTURE_FLAG,pay_types,bal,CONTRACT_STATUS,TOTAL_TERM,OVERDUE_DAYS,CONTRACT_ID,CUSTOMER_NUMBER,...,del_60_run_9,del_90_run_9,del_x_run_6,del_30_run_6,del_60_run_6,del_90_run_6,del_x_run_3,del_30_run_3,del_60_run_3,del_90_run_3
0,944370,A,N,1,507120.0,10,60.0,0,944370,6626063,...,0,0,0,0,0,0,0,0,0,0
1,944370,A,N,2,498668.0,10,60.0,0,944370,6626063,...,0,0,0,0,0,0,0,0,0,0
2,944370,A,N,2,490216.0,10,60.0,0,944370,6626063,...,0,0,0,0,0,0,0,0,0,0
3,944370,A,N,2,481764.0,10,60.0,0,944370,6626063,...,0,0,0,0,0,0,0,0,0,0
4,944370,A,N,2,473312.0,10,60.0,0,944370,6626063,...,0,0,0,0,0,0,0,0,0,0


In [7]:
# Export
df.to_parquet(
    '../data/processed/behaviour_factors.parquet',
    engine = 'pyarrow'
)